# Hand scored semantics

By [Allison Parrish](https://www.decontextualize.com), part of [Reading and writing electronic text](https://rwet.decontextualize.com/)

In this notebook, we'll work with words with hand-tagged semantic categories. Some of the details of this notebook are specific to data exported from [Doccano](https://github.com/doccano/doccano), and to the particular semantic categories that my students used in the Spring 2026 section of Reading and Writing Electronic Text, but the concepts and theory are applicable to any data tagged with multiple binary categories. If you want to learn the basics of vector representations of semantics, I recommend my [Approaches to vector semantics notebook](approaches-to-vector-semantics.ipynb) notebook, or my [Understanding word vectors notebook](understanding-word-vectors.ipynb), which covers similar material but includes a more in-depth explanation of word vectors based on distributional semantics.

# Install some libraries

We'll need a few additional libraries that we haven't installed yet in class:

In [ ]:
%pip install simpleneighbors scikit-learn numpy

[Simple Neighbors](https://github.com/aparrish/simpleneighbors) is a library for doing approximate nearest neighbor search with a friendly interface. [NumPy](https://numpy.org) is a popular Python library for doing math quickly on vectors and matrices. [Scikit-learn](https://scikit-learn.org/stable/) is a Python library that implements a number of old-school machine learning techniques, and includes some other helpful utilities.

# Load the data

I sent out a file with our tagged data in `JSONL` format, much like the data we worked with in the [part-of-speech tagging notebook](doccano-pos-tags.ipynb). Load it like so (you may need to change the string supplied to the `open` function to match the name of the file I actually sent):

In [1]:
lines = open("./semantic-categories-2026-04-09.jsonl").readlines()  # read each line into a list of strings

Taking a look at the file, we can see that it contains a number encoded JSON objects, one per line. Each line represents *an act of tagging a word* undertaken by a member of the class. The `text` field contains the word in question, and the `label` field contains a list of all categories that were tagged on the word. (The `label` list may be empty if that particular word wasn't assigned to a tagger, or if the tagger neglected to tag it.)

In [2]:
lines[:10]

['{"id":1418,"text":"excuse","label":["cold"],"Comments":[]}\n',
 '{"id":1419,"text":"kindred","label":[],"Comments":[]}\n',
 '{"id":1420,"text":"file","label":[],"Comments":[]}\n',
 '{"id":1421,"text":"surface","label":["academic","round","tangible"],"Comments":[]}\n',
 '{"id":1422,"text":"Grease","label":["outlandish","tangible","warm"],"Comments":[]}\n',
 '{"id":1423,"text":"reconcilement","label":[],"Comments":[]}\n',
 '{"id":1424,"text":"laughter","label":[],"Comments":[]}\n',
 '{"id":1425,"text":"passion","label":["emotive"],"Comments":[]}\n',
 '{"id":1426,"text":"DAY","label":["organic","round","warm"],"Comments":[]}\n',
 '{"id":1427,"text":"genius","label":[],"Comments":[]}\n']

There are a number of ways that we could transform this data into useful data structures, depending on what we want to do with it. For now, what I want to create is a data structure that looks kind of like this (the words and tag counts I'm using are placeholders, not reflective of the actual data):

    {
        'surface': {'academic': 2, 'round': 1, 'tangible': 3},
        'day': {'organic': 7, 'round': 12, 'warm': 2},
        'kitten': {'organic': 3, 'warm': 3},
        ...
    }

That is, a dictionary whose keys are the tagged words, and whose values are dictionaries that map semantic categories (like `organic`, `warm`, `tangible`) to the number of times that the word was tagged with that category.

We'll do this in two steps. In the first step, we'll create a dictionary whose keys are words from the corpus, and whose values are the list of *all tagged categories* from the data. To accomplish this, we'll use our old friend `defaultdict(list)`, which is a kind of dictionary that will automatically add a key with an empty list when we first reference it, regardless of whether or not the key already exists.

In [3]:
from collections import defaultdict, Counter
import json

# create the dictionary (unknown keys default to an empty list)
cats = defaultdict(list)

for item in lines:  # lines is a list of strings w/encoded JSON objects\
    
    obj = json.loads(item)  # decode the JSON
    
    if len(obj['label']) > 0:
        # optional: merge variations in case (e.g. DAY and day will be counted together)
        lc_text = obj['text'].lower()
        # the extend function appends the contents of one list to another
        cats[lc_text].extend(obj['label'])

The resulting dictionary matches the description above. For example:

In [4]:
cats['water']

['cold', 'organic', 'cold', 'organic', 'tangible']

We can see a subset of key/value pairs using the `.items()` method of the dictionary:

In [5]:
list(cats.items())[:10]

[('excuse', ['cold', 'emotive', 'organic', 'sharp', 'tangible']),
 ('surface', ['academic', 'round', 'tangible']),
 ('grease', ['outlandish', 'tangible', 'warm', 'tangible', 'warm']),
 ('passion', ['emotive']),
 ('day', ['organic', 'round', 'warm', 'organic']),
 ('sensation', ['emotive', 'organic', 'round', 'warm']),
 ('man', ['emotive', 'tangible']),
 ('education', ['academic', 'youthful', 'academic']),
 ('water', ['cold', 'organic', 'cold', 'organic', 'tangible']),
 ('inch', ['cold'])]

In the second step, we'll use the `Counter` class to turn these lists into counts of the number of times each tag occurs. By way of review, recall that `Counter` takes a list as parameter, and evaluates to an object that sums the number of times that each unique item in the list occurs. We can use that object like a dictionary:

In [6]:
x = Counter(['the', 'cat', 'in', 'the', 'cat', 'hat'])
print(x)
print(x['cat'])

Counter({'the': 2, 'cat': 2, 'in': 1, 'hat': 1})
2


The loop below iterates over the key/value pairs in the dictionary we made above, and creates a new dictionary with `Counter` objects as values, instead of a list of strings:

In [7]:
cats_counted = {}
for key, val in cats.items():
    cats_counted[key] = Counter(val)

Now we can see how many times each word was tagged with a particular category:

In [8]:
cats_counted['water']

Counter({'cold': 2, 'organic': 2, 'tangible': 1})

And use the `.items()` method to poke around a bit:

In [9]:
list(cats_counted.items())[:10]

[('excuse',
  Counter({'cold': 1, 'emotive': 1, 'organic': 1, 'sharp': 1, 'tangible': 1})),
 ('surface', Counter({'academic': 1, 'round': 1, 'tangible': 1})),
 ('grease', Counter({'tangible': 2, 'warm': 2, 'outlandish': 1})),
 ('passion', Counter({'emotive': 1})),
 ('day', Counter({'organic': 2, 'round': 1, 'warm': 1})),
 ('sensation', Counter({'emotive': 1, 'organic': 1, 'round': 1, 'warm': 1})),
 ('man', Counter({'emotive': 1, 'tangible': 1})),
 ('education', Counter({'academic': 2, 'youthful': 1})),
 ('water', Counter({'cold': 2, 'organic': 2, 'tangible': 1})),
 ('inch', Counter({'cold': 1}))]

> Exercise: Write a single loop that performs the JSON parsing and the conversion of the tagged categories to a `Counter` object. You may need to look at the `Counter`'s `.update()` method.

## Sorting and searching

At this point, we have a data structure that will let us search and sort the data set fairly easily. For example, to find words that have been tagged a particular number of times:

In [10]:
[key for key, val in cats_counted.items() if val['academic'] > 0 and val['youthful'] > 0]

['education',
 'apollo',
 'ignorant',
 'googler',
 'will',
 'army',
 'event',
 'merit',
 'school',
 'exercises',
 'attempts',
 'creators',
 'art',
 'greatest',
 'casual',
 'provincial',
 'origin',
 'attachment',
 'capacity',
 'outspokenness',
 'coincidence',
 'talent',
 'ideas',
 'unfair',
 'efficient',
 'characteristic',
 'lithe']

Sort the words according to their emotiveness:

In [11]:
sorted(cats_counted.items(),
       key=lambda pair: pair[1]['emotive'] / (pair[1].total()+1),
       reverse=True)[:12]

[('chuckle', Counter({'emotive': 2})),
 ('truth', Counter({'emotive': 2})),
 ('feeling', Counter({'emotive': 3, 'tangible': 1})),
 ('passion', Counter({'emotive': 1})),
 ('parent', Counter({'emotive': 1})),
 ('lust', Counter({'emotive': 2, 'organic': 1})),
 ('other', Counter({'emotive': 2, 'round': 1})),
 ('disappointment', Counter({'emotive': 1})),
 ('fate', Counter({'emotive': 3, 'outlandish': 2})),
 ('temper', Counter({'emotive': 1})),
 ('dad', Counter({'emotive': 1})),
 ('sorrow', Counter({'emotive': 1}))]

# Converting to vectors

A vector representation of this data would look like a spreadsheet where each row represents a word, and each column represents a semantic category. The value in a particular cell would have the *number of times* that word was tagged with the semantic category for that column. In other words, we want to convert a `Counter` object like this:

    Counter({'cold': 2, 'organic': 2, 'tangible': 1})

to a list that looks like this:

    [0, 2, 0, 2, 0, 0, 0, 1, 0, 0]

... assuming that the column at index 1 is the column for *cold*, the column at index 3 is the column for *organic*, and the column at index 7 is the column for *tangible*. If the word does not have a tag in the corresponding category, the value in that column is zero.

But how do we assign categories to columns? The exact order of the columns doesn't matter; we just need to make sure that every row's column assignment matches.

Get a list of unique categories:

In [12]:
unique_features = set()
for key, val in cats.items():
    unique_features.update(val)

In [13]:
unique_features

{'academic',
 'cold',
 'emotive',
 'organic',
 'outlandish',
 'round',
 'sharp',
 'tangible',
 'warm',
 'youthful'}

In [14]:
cols = list(unique_features)
print(cols)

['cold', 'sharp', 'organic', 'round', 'youthful', 'emotive', 'academic', 'tangible', 'warm', 'outlandish']


Map from the index back to the column name:

In [15]:
# a dictionary comprehension??
name2idx = {name: i for i, name in enumerate(cols)}

In [16]:
name2idx

{'cold': 0,
 'sharp': 1,
 'organic': 2,
 'round': 3,
 'youthful': 4,
 'emotive': 5,
 'academic': 6,
 'tangible': 7,
 'warm': 8,
 'outlandish': 9}

Now make a new dictionary, mapping words to their vectors:

In [17]:
cats_vectorized = {}
for word, counts in cats_counted.items():
    vector = [counts[feat] for feat in cols]
    cats_vectorized[word] = vector

In [18]:
list(cats_vectorized.items())[:10]

[('excuse', [1, 1, 1, 0, 0, 1, 0, 1, 0, 0]),
 ('surface', [0, 0, 0, 1, 0, 0, 1, 1, 0, 0]),
 ('grease', [0, 0, 0, 0, 0, 0, 0, 2, 2, 1]),
 ('passion', [0, 0, 0, 0, 0, 1, 0, 0, 0, 0]),
 ('day', [0, 0, 2, 1, 0, 0, 0, 0, 1, 0]),
 ('sensation', [0, 0, 1, 1, 0, 1, 0, 0, 1, 0]),
 ('man', [0, 0, 0, 0, 0, 1, 0, 1, 0, 0]),
 ('education', [0, 0, 0, 0, 1, 0, 2, 0, 0, 0]),
 ('water', [2, 0, 2, 0, 0, 0, 0, 1, 0, 0]),
 ('inch', [1, 0, 0, 0, 0, 0, 0, 0, 0, 0])]

# Nearest neighbors

In [19]:
from simpleneighbors import SimpleNeighbors

lookup = SimpleNeighbors(len(cols), metric="angular")
for word, vec in cats_vectorized.items():
    lookup.add_one(word, vec)
lookup.build()

In [20]:
import random

In [21]:
random.sample(lookup.corpus, 12)

['dialogue',
 'radiators',
 'smallnesses',
 'cuff buttons',
 'alexandrian',
 'philosophies',
 'capacity',
 'categories',
 'just',
 'origin',
 'dialectical',
 'plow']

In [22]:
word = random.choice(lookup.corpus)
print(word, cats_counted[word])
print(lookup.neighbors(word, 8))

minutes Counter({'organic': 1})
['ove', 'web', 'minutes', 'innate', 'bald', 'nature', 'energy', 'dialogue']


In [23]:
cols

['cold',
 'sharp',
 'organic',
 'round',
 'youthful',
 'emotive',
 'academic',
 'tangible',
 'warm',
 'outlandish']

In [24]:
lookup.nearest([0, 0, 1, 0, 0, 0, 1, 0, 0, 0])

['area',
 'whatever',
 'matter',
 'barbarians',
 'astronomy',
 'literary',
 'diffuse',
 'voluntary',
 'gong',
 'coincidence',
 'ideas',
 'existence']

In [25]:
def make_vector(**kwargs):
    vec = [0] * len(cols)
    for k, v in kwargs.items():
        vec[name2idx[k]] = v
    return vec

In [26]:
make_vector(emotive=1, sharp=1)

[0, 1, 0, 0, 0, 1, 0, 0, 0, 0]

In [27]:
lookup.nearest(make_vector(emotive=1, sharp=1, academic=3))

['application',
 'allusion',
 'powerpoint',
 'indication',
 'sentence',
 'universal',
 'interview',
 'section',
 'output',
 'indecency',
 'master',
 'aspect']

# Exercises

* Using the part of speech tagging data, recategorize these words to their part of speech. Use these to generate text using (e.g.) all *emotive* adjectives.
* Use [tfidf](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html#sklearn.feature_extraction.text.TfidfTransformer) to normalize the vector values, accounting for the overall frequency of particular categories.
* Do dimensional reduction and visualization of the word vectors with [TSNE](https://scikit-learn.org/stable/search.html?q=tsne)
* Add synthetic categories to the data, e.g., a *short* category for words with fewer than *n* letters, and a *long* category for words with *n* letters or more. Does this help/hinder the "feel" of similarity lookups?